# 实验4：CANN算子开发

## 一、实验目的

1. 理解昇腾（Ascend）AI处理器的底层硬件架构及CANN（Compute Architecture for Neural Networks）异构计算架构的基本原理。
2. 掌握基于aclnn接口开发自定义算子的完整流程，包括算子原型定义、工程生成、代码实现、编译部署与测试验证。
3. 具备使用aclnn接口进行单算子API调用的能力，能够独立完成自定义算子的开发、调试与性能优化。
4. 理解aclnn两段式算子调用机制、内存管理、流同步等核心概念，为后续AI模型开发与优化奠定基础。


## 二、实验说明

### 2.1 实验背景

CANN（Compute Architecture for Neural Networks）是华为面向AI场景推出的异构计算架构，旨在为昇腾AI处理器提供高效、易用的编程框架。在CANN生态中，算子（Operator）是构成神经网络模型的基础计算单元。

昇腾CANN算子开发经历了从TBE/TIK（基于Python DSL）到 Ascend C（基于C++ DSL）和PyPTO（Python Parallel Tensor/Tile Operation）的演进过程。TIK类似于“汇编级Python”，允许开发者精细控制UB/L1/GM的内存布局、数据搬运行为和AI Core指令调度，适合追求极致性能的场景；TBE DSL则通过自动调度机制大幅降低开发门槛，让开发者专注于算子数学逻辑，由系统自动完成调度与底层优化。

Ascend C 是 CANN 针对算子开发场景推出的编程语言，原生支持C和C++标准规范，兼具开发效率和运行性能。基于Ascend C 编写的算子程序，通过编译器编译和运行时调度，运行在昇腾AI处理器上。使用Ascend C，开发者可以基于昇腾AI硬件，高效的实现自定义的创新算法。PyPTO则是CANN新推出的一款面向AI加速器的高效编程框架，旨在简化算子开发流程，同时保持高性能计算能力，它提供了PyPTO Tensor与PyPTO Pro两种编程方式。

为了降低算子开发门槛、提升开发效率，CANN提供了aclnn（Ascend Operator Library Neural Network）接口——一套基于C语言的Level2层API，允许开发者以“单算子API执行”的方式直接调用算子，无需提供IR（Intermediate Representation）定义。

无论是基于Ascend C自定义的算子还是CANN内置算子，均可通过aclnn接口实现调用。aclnn接口的设计目标是让开发者像使用智能打印机一样调用算子——只需关注输入数据和取回结果，中间的复杂过程（如内存管理、内核调度、硬件适配等）由接口自动完成。

### 2.2 实验环境准备

本实验基于CANNLab实验环境，采用**开发环境与运行环境合设**的场景，即带AI处理器的机器既作为开发环境又作为运行环境。

#### 2.2.1 硬件环境
- 昇腾AI处理器（如Atlas 800I A2推理产品、Atlas A2训练系列产品等）
- 可通过 `npu-smi info` 命令查询芯片型号（Chip Name），获取 `<soc_version>` 参数

#### 2.2.2 软件环境
- **驱动与固件**：昇腾AI处理器驱动及固件
- **CANN软件包**：
  - `Ascend-cann-toolkit`：开发套件，提供编译工具链和头文件
  - `Ascend-cann-kernels`：算子二进制软件包
- **操作系统**：推荐openEuler或Ubuntu

#### 2.2.3 环境配置步骤

**步骤1：安装CANN软件包**

从CANN下载页面获取并安装对应版本的CANN软件包。

**步骤2：配置环境变量**

安装完成后，需配置以下环境变量才能正常使用CANN功能：
```bash
# 加载CANN环境变量（路径按实际安装位置替换）
source /usr/local/Ascend/ascend-toolkit/latest/set_env.sh
# 显式指定CANN路径
export CANN_PATH=$ASCEND_TOOLKIT_PATH
```

**步骤3：验证环境**

执行以下命令验证CANN环境是否配置成功：
```bash
python3 -c "import acl; print(acl.get_soc_name())"
```
若能正常输出芯片型号，则表示环境配置成功。

## 三、实验任务

### 3.1 任务描述

本实验要求基于aclnn接口开发一个自定义算子（以高性能 MatMul 算子为例），完成从算子定义、工程生成、代码实现到编译测试的完整流程。具体任务包括：

1. 编写算子原型定义JSON文件
2. 使用msOpGen工具生成算子开发工程
3. 实现算子的Host侧和Kernel侧代码
4. 编译算子工程，生成aclnn动态库
5. 编写测试程序调用自定义算子并验证结果

### 3.2 学习目标

- 掌握aclnn接口的调用机制与两段式API设计理念
- 熟悉msOpGen工具的使用方法
- 理解Ascend C算子的Host-Kernel分离架构
- 掌握算子编译、部署与测试的完整流程
- 能够独立完成自定义算子的开发与调试

## 四、任务准备

### 4.1 前置知识

#### 4.1.1 CANN架构分层

CANN采用分层架构，从底到顶包括：
- **Level0层**：基础张量操作接口和核函数接口
- **Level1层（nnopbase）** ：aclnn API所需的底层框架能力接口
- **Level2层（aclnn）** ：面向开发者的高层算子调用接口

aclnn接口本质是一类C语言API，CANN提供了算子加速库（Ascend Operator Library，简称ACL），包含一系列aclnn前缀的API。

#### 4.1.2 两段式算子调用机制

aclnn算子接口采用“两段式”设计：

**第一段**：`aclnnXxxGetWorkspaceSize` —— 获取算子执行所需的工作空间大小，并创建执行器。

**第二段**：`aclnnXxx` —— 执行实际的算子计算。

这种设计允许开发者在执行前预先分配所需的内存空间，提升执行效率。

#### 4.1.3 Ascend C算子工程结构

通过msOpGen工具生成的算子工程包含以下核心目录和文件（以AddCustom算子为例）：

```
AddCustom/
├── build.sh              # 编译入口脚本
├── cmake/
│   └── config.cmake      # 编译配置项
├── CMakeLists.txt        # 工程CMake文件
├── op_host/              # Host侧实现
│   ├── add_custom.cpp    # 算子原型注册、Tiling实现
│   └── add_custom.h      # Tiling定义
└── op_kernel/            # Kernel侧实现（AI Core上运行）
    └── add_custom.cpp    # 算子核心计算代码
```

### 4.2 实验数据准备

- 测试输入数据：随机生成的张量数据（如float16、float32类型）
- 测试用例：覆盖不同shape（1D、2D、3D）、不同数据类型的基础功能测试

## 五、基于Ascend C的aclnn算子开发步骤（以以AddCustom算子为例）

### 5.1 步骤一：编写算子原型定义JSON文件

算子原型定义文件描述了算子的输入、输出和属性信息，是msOpGen工具生成算子工程的依据。

创建 `add_custom.json` 文件，内容如下：

```json
[
  {
    "op": "AddCustom",
    "input_desc": [
      {
        "name": "x",
        "param_type": "required",
        "format": ["ND", "ND", "ND"],
        "type": ["fp16", "float", "int32"]
      },
      {
        "name": "y",
        "param_type": "required",
        "format": ["ND", "ND", "ND"],
        "type": ["fp16", "float", "int32"]
      }
    ],
    "output_desc": [
      {
        "name": "z",
        "param_type": "required",
        "format": ["ND", "ND", "ND"],
        "type": ["fp16", "float", "int32"]
      }
    ]
  }
]
```

**字段说明**：
- `op`：算子名称
- `input_desc`/`output_desc`：输入/输出描述
- `name`：参数名称
- `param_type`：参数类型（required表示必选）
- `format`：数据格式（ND表示普通张量）
- `type`：支持的数据类型列表

### 5.2 步骤二：使用msOpGen工具生成算子工程

使用msOpGen工具基于JSON文件生成算子开发工程：

```bash
${INSTALL_DIR}/python/site-packages/bin/msopgen gen \
  -i $HOME/sample/add_custom.json \
  -c ai_core-<soc_version> \
  -lan cpp \
  -out $HOME/sample/AddCustom \
  -f aclnn
```

**参数说明**：
- `${INSTALL_DIR}`：CANN软件安装路径（如 `/usr/local/Ascend/ascend-toolkit/latest`）
- `-i`：算子原型定义JSON文件路径
- `-c`：指定AI处理器型号（`ai_core-<soc_version>`），`<soc_version>`通过 `npu-smi info` 查询获取
- `-lan`：开发语言（cpp表示基于Ascend C使用C/C++）
- `-out`：生成工程的输出路径
- `-f`：框架类型（aclnn表示生成简易工程）

执行完成后，在 `$HOME/sample/AddCustom` 目录下生成完整的算子工程。

### 5.3 步骤三：实现算子代码

#### 5.3.1 Host侧实现（op_host/add_custom.cpp）

Host侧代码负责算子原型注册、Tiling策略定义和形状推导。主要工作包括：

1. **注册算子原型**：使用框架提供的宏注册算子输入输出信息
2. **实现Tiling策略**：根据输入张量形状计算数据分块策略，优化AI Core上的并行计算
3. **实现形状推导**：根据输入形状推导输出张量形状

#### 5.3.2 Kernel侧实现（op_kernel/add_custom.cpp）

Kernel侧代码是在AI Core上实际执行计算的代码，使用Ascend C编程框架编写：

```cpp
// 示例：AddCustom算子的Kernel实现框架
// 实际代码需根据生成的模板填充
#include "add_custom.h"

extern "C" __global__ __aicore__ void add_custom(
    GM_ADDR x, GM_ADDR y, GM_ADDR z, 
    GM_ADDR tiling, GM_ADDR workspace
) {
    // 获取当前计算的核心ID
    // 根据Tiling参数进行数据分块
    // 执行向量加法：z = x + y
    // 将结果写回全局内存
}
```

### 5.4 步骤四：编译算子工程

在算子工程目录下执行编译命令：

```bash
cd $HOME/sample/AddCustom
./build.sh
```

编译过程将：
1. 编译Host侧和Kernel侧代码
2. 生成算子静态库
3. **自动生成aclnn调用实现代码和头文件（aclnn_*.h）**
4. **链接算子静态库，生成aclnn动态库（libcust_opapi.so）** 

生成的动态库支持后续的单算子API执行方式（aclnn）的算子调用。


### 5.5 步骤五：编写测试程序

创建测试程序 `test_add.cpp`，调用自定义算子的aclnn接口。

#### 5.5.1 测试程序核心步骤

**步骤1：包含头文件**

```cpp
#include <iostream>
#include <vector>
#include "acl/acl.h"
#include "aclnn/aclnn_add_custom.h"  // 自动生成的aclnn接口头文件
```

**步骤2：初始化ACL资源**

```cpp
// 初始化ACL
aclInit(nullptr);
// 申请设备
aclrtSetDevice(deviceId);
// 创建上下文
aclrtCreateContext(&context, deviceId);
// 创建流
aclrtCreateStream(&stream);
```

**步骤3：准备输入数据并创建aclTensor**

```cpp
// 准备Host侧数据
std::vector<float> hostX = {1.0, 2.0, 3.0};
std::vector<float> hostY = {4.0, 5.0, 6.0};
std::vector<float> hostZ(3);

// 申请Device内存
aclrtMalloc(&devX, size, ACL_MEM_MALLOC_HUGE_FIRST);
aclrtMalloc(&devY, size, ACL_MEM_MALLOC_HUGE_FIRST);
aclrtMalloc(&devZ, size, ACL_MEM_MALLOC_HUGE_FIRST);

// 拷贝数据到Device
aclrtMemcpy(devX, size, hostX.data(), size, ACL_MEMCPY_HOST_TO_DEVICE);
aclrtMemcpy(devY, size, hostY.data(), size, ACL_MEMCPY_HOST_TO_DEVICE);

// 创建aclTensor
aclTensor* tensorX = aclCreateTensor(shape, dims, dataType, format, devX);
aclTensor* tensorY = aclCreateTensor(shape, dims, dataType, format, devY);
aclTensor* tensorZ = aclCreateTensor(shape, dims, dataType, format, devZ);
```

**步骤4：两段式调用aclnn接口**

```cpp
// 第一段：获取workspace大小
uint64_t workspaceSize = 0;
aclOpExecutor* executor = nullptr;
aclnnStatus ret = aclnnAddCustomGetWorkspaceSize(
    tensorX, tensorY, tensorZ, &workspaceSize, &executor);

// 申请workspace内存
void* workspace = nullptr;
if (workspaceSize > 0) {
    aclrtMalloc(&workspace, workspaceSize, ACL_MEM_MALLOC_HUGE_FIRST);
}

// 第二段：执行算子
ret = aclnnAddCustom(workspace, workspaceSize, executor, stream);

// 同步流，等待算子执行完成
aclrtSynchronizeStream(stream);
```

**步骤5：取回结果并验证**

```cpp
// 将结果从Device拷贝回Host
aclrtMemcpy(hostZ.data(), size, devZ, size, ACL_MEMCPY_DEVICE_TO_HOST);

// 与CPU计算结果对比验证
for (size_t i = 0; i < hostZ.size(); i++) {
    float expected = hostX[i] + hostY[i];
    if (fabs(hostZ[i] - expected) > 1e-5) {
        std::cout << "验证失败 at index " << i << std::endl;
    }
}
```

**步骤6：释放资源**

```cpp
aclrtFree(devX);
aclrtFree(devY);
aclrtFree(devZ);
aclrtFree(workspace);
aclDestroyTensor(tensorX);
aclDestroyTensor(tensorY);
aclDestroyTensor(tensorZ);
aclrtDestroyStream(stream);
aclrtDestroyContext(context);
aclrtResetDevice(deviceId);
aclFinalize();
```


#### 5.5.2 编译测试程序

创建 `CMakeLists.txt` 文件，配置编译选项：

```cmake
cmake_minimum_required(VERSION 3.14)
project(ACLNN_EXAMPLE)

add_compile_options(-std=c++11)

set(CMAKE_RUNTIME_OUTPUT_DIRECTORY "./bin")
set(CMAKE_CXX_FLAGS_DEBUG "-fPIC -O0 -g -Wall")
set(CMAKE_CXX_FLAGS_RELEASE "-fPIC -O2 -Wall")

add_executable(opapi_test test_add.cpp)

# 设置CANN路径
if(NOT "$ENV{ASCEND_CUSTOM_PATH}" STREQUAL "")
    set(ASCEND_PATH $ENV{ASCEND_CUSTOM_PATH})
else()
    set(ASCEND_PATH "/usr/local/Ascend/ascend-toolkit/latest")
endif()

set(INCLUDE_BASE_DIR "${ASCEND_PATH}/include")
include_directories(
    ${INCLUDE_BASE_DIR}
    ${INCLUDE_BASE_DIR}/aclnn
)

# 链接库文件
target_link_libraries(opapi_test PRIVATE
    ${ASCEND_PATH}/lib64/libascendcl.so
    ${ASCEND_PATH}/lib64/libnnopbase.so
    ${ASCEND_PATH}/lib64/libopapi.so
)

install(TARGETS opapi_test DESTINATION ${CMAKE_RUNTIME_OUTPUT_DIRECTORY})
```

编译测试程序：
```bash
mkdir build && cd build
cmake .. && make
```

#### 5.5.3 运行测试

```bash
./bin/opapi_test
```

## 六、任务实施

### 步骤一：导入必要的库

In [1]:
import os 
import numpy as np
import acl

print("导入库成功！")

导入库成功！


### 步骤二：定义常量和 ACL 初始化

In [2]:
device_id = 0

# 定义矩阵维度 M, K, N (A[M, K] * B[K, N] = C[M, N])
M, K, N = 16, 32, 64

print("开始初始化 ACL...")
acl.init()
acl.rt.set_device(device_id)
context, ret = acl.rt.create_context(device_id)
print("ACL 初始化成功!")

开始初始化 ACL...
ACL 初始化成功!


### 步骤三：准备 Host (CPU) 侧数据

In [3]:
input_A_host = np.full((M, K), 1.0, dtype=np.float32)
input_B_host = np.full((K, N), 2.0, dtype=np.float32)
output_C_host = np.empty((M, N), dtype=np.float32)

print(f"Host 侧数据准备完毕：")
print(f"输入 A 形状: {input_A_host.shape}")
print(f"输入 B 形状: {input_B_host.shape}")
print(f"输出 C 形状: {output_C_host.shape}")

Host 侧数据准备完毕：
输入 A 形状: (16, 32)
输入 B 形状: (32, 64)
输出 C 形状: (16, 64)


### 步骤四：准备 Device (NPU) 侧内存

In [10]:
ACL_MEM_MALLOC_HUGE_FIRST = 0
ACL_MEMCPY_HOST_TO_DEVICE = 1
ACL_MEMCPY_DEVICE_TO_HOST = 2

size_A = M * K * 4
size_B = K * N * 4
size_C = M * N * 4

input_A_dev, ret = acl.rt.malloc(size_A, ACL_MEM_MALLOC_HUGE_FIRST)
input_B_dev, ret = acl.rt.malloc(size_B, ACL_MEM_MALLOC_HUGE_FIRST)
output_C_dev, ret = acl.rt.malloc(size_C, ACL_MEM_MALLOC_HUGE_FIRST)

src_ptr = acl.util.bytes_to_ptr(input_A_host.tobytes())
acl.rt.memcpy(input_A_dev, size_A, src_ptr, size_A, ACL_MEMCPY_HOST_TO_DEVICE)

src_ptr = acl.util.bytes_to_ptr(input_B_host.tobytes())
acl.rt.memcpy(input_B_dev, size_B, src_ptr, size_B, ACL_MEMCPY_HOST_TO_DEVICE)

print("Device 内存申请并拷贝成功！")

Device 内存申请并拷贝成功！


### 步骤五：准备描述张量 (TensorDesc)

In [13]:
ACL_FLOAT = 0
ACL_FORMAT_ND = 0

desc_A = acl.create_tensor_desc(ACL_FLOAT, [M, K], ACL_FORMAT_ND)
desc_B = acl.create_tensor_desc(ACL_FLOAT, [K, N], ACL_FORMAT_ND)
desc_C = acl.create_tensor_desc(ACL_FLOAT, [M, N], ACL_FORMAT_ND)

print("张量描述 (TensorDesc) 创建成功。")

张量描述 (TensorDesc) 创建成功。


### 步骤六：准备数据 Buffer

In [14]:
buffer_A = acl.create_data_buffer(input_A_dev, size_A)
buffer_B = acl.create_data_buffer(input_B_dev, size_B)
buffer_C = acl.create_data_buffer(output_C_dev, size_C)

print("数据 Buffer (DataBuffer) 创建成功。")

数据 Buffer (DataBuffer) 创建成功。


### 步骤七：执行 ACLNN 算子 

In [19]:
print("开始执行 ACLNN MatMul 算子...")

stream, ret = acl.rt.create_stream()
handle, ret = acl.op.create_handle("Matmul", [desc_A, desc_B], [desc_C], 0)
ret = acl.op.execute_with_handle(handle, [buffer_A, buffer_B], [buffer_C], stream)
ret = acl.rt.synchronize_stream(stream)

print("MatMul 算子执行成功！")

: 

### 步骤八：获取结果并验证结果的正确性

In [30]:
result_buffer, ret = acl.rt.malloc_host(size_C)
ret = acl.rt.memcpy(result_buffer, size_C, buffer_C, size_C, ACL_MEMCPY_DEVICE_TO_HOST)
result_bytes = acl.util.ptr_to_bytes(result_buffer, size_C)
result = np.frombuffer(result_bytes, dtype=np.float32)

verify_C_host = np.dot(input_A_host, input_B_host)

print("\n======== 结果验证 ========")
print(f"Numpy (CPU) 计算结果 (预期): {verify_C_host[0, 0]}")
print(f"ACLNN (NPU) 计算结果 (实际): {result[0]}")
print(f"ACLNN (NPU) 计算结果 (最后一个元素): {result[(M-1) * (N-1)]}")
print("==========================\n")

if np.allclose(result.reshape(verify_C_host.shape), verify_C_host):
    print("结果正确！NPU 计算结果与 Numpy 一致！")
else:
    print("结果错误！NPU 计算结果与 Numpy 不符！")


======== 结果验证 ========
Numpy (CPU) 计算结果 (预期): 64.0
ACLNN (NPU) 计算结果 (实际): -2.052734375
ACLNN (NPU) 计算结果 (最后一个元素): 0.0

结果错误！NPU 计算结果与 Numpy 不符！


### 步骤九：清理所有资源

In [1]:
print("开始清理所有 ACL 资源...")

# 释放内存 (Host 和 Device)
acl.rt.free(input_A_dev)
acl.rt.free(input_B_dev)
acl.rt.free(output_C_dev)
# (Host 侧 numpy 数组会被自动回收，无需 free_host)

# 销毁 DataBuffer
acl.destroy_data_buffer(buffer_A)
acl.destroy_data_buffer(buffer_B)
acl.destroy_data_buffer(buffer_C)

# 销毁 TensorDesc
acl.destroy_tensor_desc(desc_A)
acl.destroy_tensor_desc(desc_B)
acl.destroy_tensor_desc(desc_C)

acl.rt.destroy_stream(stream)
acl.op.destroy_handle(handle)

# 销毁 Context 和 Device
acl.rt.destroy_context(context)
acl.rt.reset_device(device_id)

# ACL 去初始化
acl.finalize()

print("清理完毕。")

开始清理所有 ACL 资源...


NameError: name 'acl' is not defined

## 七、任务拓展

### 7.1 基于Ascend C实现matmul算子

按照上面 **“基于Ascend C的aclnn算子开发步骤（以以AddCustom算子为例）”** 给出的步骤，用Ascend C实现基于aclnn的matmul算子。

### 7.2 性能优化

- **Tiling策略优化**：根据输入shape和AI Core数量优化数据分块策略，提升计算并行度
- **内存复用**：减少不必要的内存分配和拷贝操作
- **异步执行**：充分利用流（Stream）机制实现计算与数据传输的重叠

### 7.3 多算子融合

在掌握单算子开发的基础上，可以尝试开发融合算子（如 `AddMatMul`），将多个计算操作融合为一个算子，减少内核启动开销和内存访问次数。

### 7.4 PyTorch框架集成

将开发的aclnn算子通过Pybind11绑定到Python生态，使自定义算子能够像PyTorch原生算子一样在Python侧调用。

### 7.5 算子调试技巧

- 使用 `printf` 调试时需注意AI Core上的输出限制
- 利用CANN提供的日志系统查看算子执行详细信息
- 使用 `npu-smi` 工具监控NPU资源使用情况


## 八、实验总结

### 8.1 核心知识点回顾

通过本实验，我们系统学习了基于aclnn接口的CANN算子开发全流程：

1. **aclnn接口体系**：理解了aclnn作为Level2层接口的设计理念——以简洁的API屏蔽底层硬件差异，实现“单算子API执行”。

2. **两段式调用机制**：掌握了 `GetWorkspaceSize` + `Execute` 的两段式API设计，理解了workspace预分配对性能的意义。

3. **算子开发工具链**：熟悉了msOpGen工具从JSON原型定义到完整工程生成的自动化流程。

4. **Host-Kernel分离架构**：理解了Host侧（原型注册、Tiling、形状推导）和Kernel侧（AI Core计算）的分工与协作。

5. **完整的开发闭环**：经历了从算子定义→工程生成→代码实现→编译部署→测试验证的完整开发流程。

### 8.2 关键注意事项

- **环境配置**：必须正确安装 `Ascend-cann-toolkit` 和 `Ascend-cann-kernels` 软件包，并配置环境变量
- **SOC版本**：创建算子工程时必须指定正确的 `<soc_version>`，否则生成的算子无法在目标硬件上运行
- **内存管理**：注意Host与Device内存的分配与释放，避免内存泄漏
- **流同步**：算子执行后必须调用 `aclrtSynchronizeStream` 确保计算完成后再取回结果
- **错误处理**：检查每个aclnn API的返回值，及时定位问题

### 8.3 实验心得

aclnn接口大大降低了昇腾平台算子开发的门槛。通过本实验，我们不仅掌握了具体的开发技能，更深入理解了异构计算架构中“易用性”与“高性能”之间的平衡设计——aclnn在提供简洁API的同时，通过两段式设计、异步执行、Tiling优化等机制确保了接近手写底层代码的性能。这为后续在昇腾平台上进行模型适配、性能优化和自定义算子开发奠定了坚实的基础。